In [0]:
# Notebook autonome : il porte ses propres %run et se lance seul.
# Relancés par main_translations, ils sont sans effet de bord (idempotents).

In [0]:
%run ./env

In [0]:
%run ./python_libraries

In [0]:
%run ../delta_function

In [0]:
%run ./translation_function

# build_nomenclatures

Construit les **dimensions de nomenclature** dans le schéma `common` : une ligne
par entité métier, avec toutes ses colonnes source.

## Pourquoi elles sont nécessaires

Sans elles, la relation Power BI entre les faits et une table de traduction est
**plusieurs-à-plusieurs** : `dim_batches_specifications[id_good_variety]` n'est
pas unique (plusieurs batches par variété) et `dim_trad_variety[id_good_variety]`
non plus (une ligne par langue). Deux côtés « plusieurs ».

La nomenclature rétablit un schéma en étoile classique :

```
dim_batches_specifications  --*→1--  dim_variety  --1←*--  dim_trad_variety
                                   (1 ligne/variété)      (4 lignes/variété)
```

Chaque relation a un côté « 1 » identifié, et le `code` de la nomenclature sert
de colonne de tri : l'ordre des libellés reste stable d'une langue à l'autre.

## Conventions retenues

- **Toutes les colonnes source sont conservées**, sans renommage. Ajouter une
  colonne plus tard imposerait un `ALTER TABLE` : autant tout prendre maintenant,
  ces tables font quelques milliers de lignes.
- **`filter(deleted == False)`** : seules les entités actives entrent dans le
  modèle. Le rapport n'a pas à connaître les entités supprimées, et le filtrage
  est fait une fois ici plutôt qu'à chaque import Power BI.
- **`mode="full"`**, et non `execution_mode`. Le mode `update` ne supprime
  jamais : une entité passée à `deleted = true` disparaît du DataFrame filtré, le
  MERGE ne la rencontre donc plus et sa ligne reste indéfiniment dans la table
  cible, figée avec `deleted = false`. Le libellé obsolète continuerait
  d'apparaître dans les slicers du rapport. Ces tables font quelques centaines de
  lignes : les réécrire coûte moins qu'un MERGE colonne par colonne.
- `ensure_delta_table` crée la table au premier run — `handle_table_update`
  échoue sur une table absente.

In [0]:
# Sources : les tables métier de la base PostgreSQL du front.
plants_production_lines = spark.table(f"{source_catalog}.plants_production_lines")
goods_species = spark.table(f"{source_catalog}.goods_species")
goods_varieties = spark.table(f"{source_catalog}.goods_varieties")
requirement_specifications = spark.table(f"{source_catalog}.requirement_specifications")
parameters_production_types = spark.table(f"{source_catalog}.parameters_production_types")
parameters_variables = spark.table(f"{source_catalog}.parameters_variables")
parameters_localizations = spark.table(f"{source_catalog}.parameters_localizations")
parameters_localization_groups = spark.table(f"{source_catalog}.parameters_localization_groups")
parameters_production_line_variables = spark.table(f"{source_catalog}.parameters_production_line_variables")
parameters_batch_note_categories = spark.table(f"{source_catalog}.parameters_batch_note_categories")

## dim_site

Colonnes reprises du notebook `dim_site` existant, **y compris son alias** :
celui-ci renomme `name` en `plant`, et c'est ce nom que les 13 slicers du
rapport interrogent. La colonne `plant` de la source, qui porte un numéro et
non un libellé, est donc conservée sous le nom `plant_code` pour éviter la
collision.

> Une différence avec ta version : **le filtre `deleted == False` est retiré**,
> comme pour les autres nomenclatures. Les lignes supprimées restent avec leur
> drapeau à jour, et le filtrage se fait à l'import Power BI.

In [0]:
dim_site = (
    plants_production_lines
    .filter(F.col("deleted") == False)
    .select(
        "id_plant_production_line",
        F.col("name").alias("plant"),
        F.col("plant").alias("plant_code"),
        "process",
        "default_production_type",
        "deleted",
        "created_at",
        "updated_at",
        "deleted_at",
        "default_batch_cycle",
        "sap_code",
        "common_code"
    )
)

current_process = "dim_site"
target_dim_site = current_catalog + "." + common_schema + "." + current_process
print(target_dim_site)

all_columns = dim_site.columns
primary_key = ['id_plant_production_line']
additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug':
    display(all_columns)
    print(additional_columns)

ensure_delta_table(dim_site, target_dim_site)

handle_table_update(
    dim_site,
    target_dim_site,
    primary_key,
    all_columns,
    additional_columns_to_check=additional_columns,
    mode="full"
)

## dim_specy

In [0]:
dim_specy = goods_species.filter(F.col("deleted") == False)

current_process = "dim_specy"
target_dim_specy = current_catalog + "." + common_schema + "." + current_process
print(target_dim_specy)

all_columns = dim_specy.columns
primary_key = ['id_good_specy']
additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug':
    display(all_columns)
    print(additional_columns)

ensure_delta_table(dim_specy, target_dim_specy)

handle_table_update(
    dim_specy,
    target_dim_specy,
    primary_key,
    all_columns,
    additional_columns_to_check=additional_columns,
    mode="full"
)

## dim_variety

La colonne `specy` porte le rattachement à l'espèce : elle permet une hiérarchie
espèce > variété dans le modèle.

In [0]:
dim_variety = goods_varieties.filter(F.col("deleted") == False)

current_process = "dim_variety"
target_dim_variety = current_catalog + "." + common_schema + "." + current_process
print(target_dim_variety)

all_columns = dim_variety.columns
primary_key = ['id_good_variety']
additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug':
    display(all_columns)
    print(additional_columns)

ensure_delta_table(dim_variety, target_dim_variety)

handle_table_update(
    dim_variety,
    target_dim_variety,
    primary_key,
    all_columns,
    additional_columns_to_check=additional_columns,
    mode="full"
)

## dim_production_type

In [0]:
dim_production_type = parameters_production_types.filter(F.col("deleted") == False)

current_process = "dim_production_type"
target_dim_production_type = current_catalog + "." + common_schema + "." + current_process
print(target_dim_production_type)

all_columns = dim_production_type.columns
primary_key = ['id_parameter_production_type']
additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug':
    display(all_columns)
    print(additional_columns)

ensure_delta_table(dim_production_type, target_dim_production_type)

handle_table_update(
    dim_production_type,
    target_dim_production_type,
    primary_key,
    all_columns,
    additional_columns_to_check=additional_columns,
    mode="full"
)

## dim_variable

In [0]:
dim_variable = parameters_variables.filter(F.col("deleted") == False)

current_process = "dim_variable"
target_dim_variable = current_catalog + "." + common_schema + "." + current_process
print(target_dim_variable)

all_columns = dim_variable.columns
primary_key = ['id_parameter_variable']
additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug':
    display(all_columns)
    print(additional_columns)

ensure_delta_table(dim_variable, target_dim_variable)

handle_table_update(
    dim_variable,
    target_dim_variable,
    primary_key,
    all_columns,
    additional_columns_to_check=additional_columns,
    mode="full"
)

## dim_requirement_specification

> Il n'existe **pas** de `requirement_specifications_translations` en base. Cette
> nomenclature sert la relation et le tri ; son libellé restera dans la langue de
> saisie. À confirmer avec le PO : oubli du front, ou champ volontairement non
> traduit ?

In [0]:
dim_requirement_specification = requirement_specifications.filter(F.col("deleted") == False)

current_process = "dim_requirement_specification"
target_dim_requirement_specification = current_catalog + "." + common_schema + "." + current_process
print(target_dim_requirement_specification)

all_columns = dim_requirement_specification.columns
primary_key = ['id_requirement_specification']
additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug':
    display(all_columns)
    print(additional_columns)

ensure_delta_table(dim_requirement_specification, target_dim_requirement_specification)

handle_table_update(
    dim_requirement_specification,
    target_dim_requirement_specification,
    primary_key,
    all_columns,
    additional_columns_to_check=additional_columns,
    mode="full"
)

## dim_localization

> La table porte une colonne **`group`** (rattachement au groupe de localisation),
> qui est un mot réservé SQL. Elle est conservée telle quelle, mais toute requête
> ultérieure devra l'échapper par des accents graves : `` `group` ``.

In [0]:
dim_localization = parameters_localizations.filter(F.col("deleted") == False)

current_process = "dim_localization"
target_dim_localization = current_catalog + "." + common_schema + "." + current_process
print(target_dim_localization)

all_columns = dim_localization.columns
primary_key = ['id_parameter_localization']
additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug':
    display(all_columns)
    print(additional_columns)

ensure_delta_table(dim_localization, target_dim_localization)

handle_table_update(
    dim_localization,
    target_dim_localization,
    primary_key,
    all_columns,
    additional_columns_to_check=additional_columns,
    mode="full"
)

## dim_localization_group

Identifiée par le groupe seul. Sa table de traduction porte une clé double
(groupe + ligne de production), mais la vérification en PROD montre qu'aucun
groupe n'a de libellé différent selon la ligne : `build_translations` ramène donc
la clé à une seule colonne, et la relation est un simple 1→*.

In [0]:
dim_localization_group = parameters_localization_groups.filter(F.col("deleted") == False)

current_process = "dim_localization_group"
target_dim_localization_group = current_catalog + "." + common_schema + "." + current_process
print(target_dim_localization_group)

all_columns = dim_localization_group.columns
primary_key = ['id_parameter_localization_group']
additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug':
    display(all_columns)
    print(additional_columns)

ensure_delta_table(dim_localization_group, target_dim_localization_group)

handle_table_update(
    dim_localization_group,
    target_dim_localization_group,
    primary_key,
    all_columns,
    additional_columns_to_check=additional_columns,
    mode="full"
)

## dim_production_line_variable

Cette table n'a **pas de colonne `code`** : le libellé n'existe que dans la table
de traduction. `variable` pointe `parameters_variables`, `production_line` pointe
`plants_production_lines`, et `displayed` permettra de masquer dans le rapport les
paramètres que le front ne veut pas exposer.

In [0]:
dim_production_line_variable = parameters_production_line_variables.filter(F.col("deleted") == False)

current_process = "dim_production_line_variable"
target_dim_production_line_variable = current_catalog + "." + common_schema + "." + current_process
print(target_dim_production_line_variable)

all_columns = dim_production_line_variable.columns
primary_key = ['id_parameter_production_line_variable']
additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug':
    display(all_columns)
    print(additional_columns)

ensure_delta_table(dim_production_line_variable, target_dim_production_line_variable)

handle_table_update(
    dim_production_line_variable,
    target_dim_production_line_variable,
    primary_key,
    all_columns,
    additional_columns_to_check=additional_columns,
    mode="full"
)

## Les 4 nomenclatures de notes de production

`parameters_batch_note_categories` sert **quatre usages** : `category_class`
porte le type (`location`, `event`, `detail`, `impact`).

On en produit **4 tables distinctes** plutôt qu'une seule, parce que
`fact_batch_note` a quatre colonnes à traduire et que Power BI n'autorise
qu'**une seule relation active** entre deux tables. Une table unique imposerait
trois relations inactives et des `USERELATIONSHIP` dans chaque mesure —
ingérable pour des colonnes posées directement sur les axes des visuels.

`category_order` donnera le tri métier, stable entre les langues.

In [0]:
if verbose_mode == 'debug':
    print("Répartition par category_class :")
    display(
        parameters_batch_note_categories
        .groupBy("category_class", "deleted").count().orderBy("category_class")
    )

In [0]:
for category_type in ["location", "event", "detail", "impact"]:
    df_type = parameters_batch_note_categories.filter(
        (F.col("category_class") == category_type) & (F.col("deleted") == False)
    )

    current_process = f"dim_batch_note_{category_type}"
    target_batch_note = current_catalog + "." + common_schema + "." + current_process
    print(target_batch_note)

    all_columns = df_type.columns
    primary_key = ['id_batch_note_category']
    additional_columns = get_additional_columns(all_columns, primary_key)

    ensure_delta_table(df_type, target_batch_note)

    handle_table_update(
        df_type,
        target_batch_note,
        primary_key,
        all_columns,
        additional_columns_to_check=additional_columns,
        mode="full"
    )

## Périmètre couvert

**13 tables** dans `common`. Chaque table de traduction a désormais son côté
« 1 » : plus aucune relation plusieurs-à-plusieurs dans le modèle.

> **À ne pas oublier côté Power BI** : ces tables contiennent les lignes
> supprimées. Filtrer `deleted = false` à l'import, sinon les libellés obsolètes
> réapparaîtront dans les slicers.